# Exploración inicial — ENLEC (lectura)

Este notebook busca entender la historia que puede contener la base ENLEC_FORM_LECTURA.csv, sin sacar conclusiones causales ni preparar un texto para publicar todavía.

## Objetivo general
- revisar la estructura real de la base;
- identificar filas, columnas, tipos, faltantes y códigos de no respuesta;
- detectar si hay pesos o factores de expansión;
- localizar variables sociodemográficas y geográficas;
- explorar patrones de lectura, frecuencia, gusto, tipo de lectura, lectura digital y razones para no leer;
- preparar la evidencia para construir la narrativa del episodio.

## Importante
- No interpretar códigos numéricos sin contrastarlos con el diccionario de variables.
- Mantener visibles los faltantes/no aplica.
- No afirmar causalidad ni sustituir lectura física por lectura digital.
- Usar factores de expansión cuando corresponda.

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd

project_root = Path.cwd().resolve().parents[1]
raw_dir = project_root / 'data' / 'raw'
file_path = Path(r'C:\Users\USUARIO\Desktop\Colombia_en_datos\episodios\005_pruebas_pisa\data\raw\ENLEC_FORM_LECTURA.csv')

print('Proyecto:', project_root)
print('Archivo:', file_path)
print('Existe:', file_path.exists())

if file_path.exists():
    df = pd.read_csv(file_path, low_memory=False)
    print('Filas:', df.shape[0])
    print('Columnas:', df.shape[1])
    display(df.head(3))
else:
    print('La base no existe en la ruta esperada.')

Proyecto: C:\Users\USUARIO\Desktop\Colombia_en_datos\episodios
Archivo: C:\Users\USUARIO\Desktop\Colombia_en_datos\episodios\005_pruebas_pisa\data\raw\ENLEC_FORM_LECTURA.csv
Existe: True
Filas: 96442
Columnas: 163


,FK_ID_FORMULARIO,ID_VIVIENDA,NUMERO_HOGAR,NUMERO_PERSONA,P1702,P1703,P1704,P1705,P1852,P1852S1,...,P1873S6,P1873S7,P1873S10,P1792S1,P1792S4,P1792S5,P1792S6,P1792S7,P1792S8,P1793
0,181,1,1,1,inf,inf,inf,inf,1,3.0,...,inf,inf,inf,1,2,1,1,1,1,3
1,181,1,1,2,inf,inf,inf,inf,2,inf,...,inf,inf,inf,1,1,1,1,1,1,1
2,181,1,1,3,1.0,3.0,2.0,inf,2,inf,...,1.0,2.0,2.0,1,1,1,1,1,1,3


## 1) Número de filas y columnas, tipos de variables y vista preliminar

Se hace una primera inspección para entender el tamaño de la base, los tipos y la estructura general.

In [5]:
if 'df' in globals():
    print('Tipos de variables:')
    print(df.dtypes.head(30).to_string())
    print('\nResumen de filas y columnas:')
    print(df.shape)
    print('\nColumnas iniciales:')
    print(df.columns[:30].tolist())
else:
    print('No hay dataset cargado.')

Tipos de variables:
FK_ID_FORMULARIO      int64
ID_VIVIENDA           int64
NUMERO_HOGAR          int64
NUMERO_PERSONA        int64
P1702               float64
P1703               float64
P1704               float64
P1705               float64
P1852                 int64
P1852S1             float64
P1852S2A1           float64
P1853                 int64
P1853S1             float64
P1853S2A1           float64
P1854                 int64
P1854S1             float64
P1854S2A1           float64
P1851                 int64
P1851S1             float64
P1851S2A1           float64
P1855                 int64
P1855S1             float64
P1855S2A1           float64
P1856                 int64
P1856S1             float64
P1856S2A1           float64
P1858                 int64
P1858S1             float64
P1858S2A1           float64
P1859                 int64

Resumen de filas y columnas:
(96442, 163)

Columnas iniciales:
['FK_ID_FORMULARIO', 'ID_VIVIENDA', 'NUMERO_HOGAR', 'NUMERO_PERSONA', 'P1702

In [6]:
if 'df' in globals():
    # Recuento de valores faltantes
    faltantes = df.isna().sum().sort_values(ascending=False)
    print('Top 20 columnas con más faltantes:')
    print(faltantes.head(20).to_string())

    # Detección de sentinelas numéricas típicas de no respuesta / no aplica
    num = df.apply(pd.to_numeric, errors='coerce')
    sentinelas = [np.inf, -1, -9, 9, 99, 98, 999, 9999, 99999]
    print('\nConteos de sentinelas numéricas:')
    for val in sentinelas:
        count = int((num == val).sum().sum())
        if count > 0:
            print(f'{val}: {count}')

    # Muestra especial: valores muy grandes del tipo 1.79769313486232e+308
    big = num.abs().max().max()
    print(f'\nMáximo absoluto observado en columnas numéricas: {big}')
    if np.isfinite(big):
        print('No se observa un valor de sentinela extremo en este primer chequeo.')
else:
    print('No hay dataset cargado.')

Top 20 columnas con más faltantes:
FK_ID_FORMULARIO    0
P1869S5             0
P1865S10            0
P1784               0
P1786               0
P1787               0
P1866S1             0
P1866S2             0
P1866S3             0
P1866S4             0
P1866S5             0
P1866S6             0
P1866S7             0
P1866S8             0
P1867               0
P1868               0
P1869S1             0
P1869S2             0
P1869S3             0
P1865S9             0

Conteos de sentinelas numéricas:
inf: 7428223
9: 24801
99: 1085
98: 7
9999: 3

Máximo absoluto observado en columnas numéricas: inf


## 2) Identificación de faltantes, no responde y no aplica

La base de ENLEC usa códigos numéricos y sentinelas. Antes de interpretar cualquier pregunta, hay que distinguir entre:
- faltante real,
- no responde,
- no aplica,
- valor omitido por el diseño del formulario.

Esto es clave para no confundir lectura con falta de información.

In [7]:
if 'df' in globals():
    # Inspección rápida de patrones no numéricos y columnas con muchísimos valores vacíos
    vacios = df.isna().sum().sort_values(ascending=False)
    print('Columnas con más vacíos:')
    print(vacios.head(15).to_string())

    # Revisión de valores muy grandes / infinitos tras coerción numérica
    num = df.apply(pd.to_numeric, errors='coerce')
    inf_count = int(np.isinf(num.to_numpy(dtype=float)).sum())
    print(f'\nCeldas con infinito tras coerción numérica: {inf_count}')

    # Si aparecen valores de tipo 1.79769313486232e+308, conviene tratarlos como no respuesta
    if inf_count > 0:
        print('Se recomienda convertir esos valores a NaN antes de análisis exploratorio.')
else:
    print('No hay dataset cargado.')

Columnas con más vacíos:
FK_ID_FORMULARIO    0
P1869S5             0
P1865S10            0
P1784               0
P1786               0
P1787               0
P1866S1             0
P1866S2             0
P1866S3             0
P1866S4             0
P1866S5             0
P1866S6             0
P1866S7             0
P1866S8             0
P1867               0

Celdas con infinito tras coerción numérica: 7428223
Se recomienda convertir esos valores a NaN antes de análisis exploratorio.


In [8]:
# Cargar el diccionario de variables (metadatos) para leer etiquetas y códigos reales.
raw_meta_files = sorted(raw_dir.glob('variables-*.csv'))
print('Archivos de metadatos:', len(raw_meta_files))
print([p.name for p in raw_meta_files[:5]])

if raw_meta_files:
    meta_frames = [pd.read_csv(f, dtype=str, keep_default_na=False) for f in raw_meta_files]
    meta = pd.concat(meta_frames, ignore_index=True)
    print('Filas en metadatos:', meta.shape[0])
    print('Columnas en metadatos:', meta.columns.tolist())
    display(meta.head(2))
else:
    print('No se encontraron archivos de metadatos en data/raw/.')

Archivos de metadatos: 0
[]
No se encontraron archivos de metadatos en data/raw/.


In [9]:
if 'meta' in globals():
    # Buscamos pesos / factores / expansión en etiquetas y nombres
    peso_mask = (
        meta['name'].astype(str).str.contains('peso|ponder|expan|factor|wgt', case=False, na=False)
        | meta['labl'].astype(str).str.contains('peso|ponder|expan|factor', case=False, na=False)
        | meta['var_wgt'].astype(str).str.contains('1|yes|true', case=False, na=False)
    )
    peso_vars = meta.loc[peso_mask, ['name', 'labl', 'var_wgt']]
    print('Variables relacionadas con pesos o expansión:')
    if peso_vars.empty:
        print('No se detectan columnas de peso explícitas en la base ni en los metadatos.')
    else:
        display(peso_vars.head(20))
else:
    print('No hay metadatos cargados.')

No hay metadatos cargados.


## 3) Factores de expansión, pesos muestrales y variables clave de diseño

La ausencia de columnas con nombres explícitos como peso o ponderador no garantiza que no haya pesos. Hay que revisarlo con el diccionario y la documentación del ENLEC.

En esta exploración inicial se busca responder:
- ¿hay una variable de peso en el formulario o en archivos auxiliares?
- ¿el archivo tiene una estructura de diseño muestral que requiere ponderación?
- ¿las cifras deben presentarse con y sin ponderación para mostrar sensibilidad del resultado?

In [11]:
# Revisión de variables demográficas y geográficas según etiquetas del diccionario.
if 'meta' in globals():
    geo_terms = ['depart', 'municip', 'zona', 'regional', 'territ', 'cabecera', 'rural', 'area']
    demo_terms = ['edad', 'sexo', 'genero', 'educ', 'escolar', 'nivel', 'hogar', 'persona', 'famil', 'jefe']

    geo_hits = meta[
        meta['labl'].astype(str).str.contains('|'.join(geo_terms), case=False, na=False)
        | meta['name'].astype(str).str.contains('|'.join(geo_terms), case=False, na=False)
    ][['name', 'labl']].drop_duplicates()

    demo_hits = meta[
        meta['labl'].astype(str).str.contains('|'.join(demo_terms), case=False, na=False)
        | meta['name'].astype(str).str.contains('|'.join(demo_terms), case=False, na=False)
    ][['name', 'labl']].drop_duplicates()

    print('Variables geográficas detectadas (primeras 30):')
    print(geo_hits.head(30).to_string(index=False))
    print('\nVariables sociodemográficas detectadas (primeras 30):')
    print(demo_hits.head(30).to_string(index=False))
else:
    print('No hay metadatos cargados.')

No hay metadatos cargados.


In [12]:
if 'meta' in globals():
    # Construcción de una lista de variables potencialmente relevantes para lectura.
    lectura_terms = [
        'leer', 'lectura', 'libro', 'libros', 'material', 'digital',
        'internet', 'noticia', 'revista', 'periódico', 'novela',
        'acad', 'documento', 'texto', 'frecuencia', 'gusto'
    ]

    reading_hits = meta[
        meta['labl'].astype(str).str.contains('|'.join(lectura_terms), case=False, na=False)
        | meta['name'].astype(str).str.contains('P17|P18|P19|P53|P10|P18|P19|P20', regex=True, na=False)
    ][['name', 'labl']].drop_duplicates()

    print('Variables relacionadas con lectura detectadas por el diccionario:')
    display(reading_hits.head(80))
else:
    print('No hay metadatos cargados.')

No hay metadatos cargados.


## 4) Dimensión A. Intensidad de lectura

Aquí se explora el número de libros leídos, distribución y presencia de valores extremos o atípicos.

Se recomienda empezar con una variable como la de número de libros leídos en los últimos 12 meses y luego confirmar su significado en el diccionario del ENLEC.

In [13]:
# Prioridades de análisis: intentar localizar la variable de número de libros leídos.
if 'meta' in globals():
    libro_expr = meta[
        meta['labl'].astype(str).str.contains('libros|leyó.*libros|en los últimos 12 meses.*libros', case=False, na=False)
        | meta['name'].astype(str).str.contains('P5381|P5380|P5384|P5390', regex=True, na=False)
    ][['name', 'labl']]
    print('Variables probablemente relacionadas con cantidad de libros:')
    display(libro_expr.drop_duplicates().head(30))
else:
    print('No hay metadatos cargados.')

No hay metadatos cargados.


In [14]:
# Selección operativa de la variable principal: si existe P5381, la exploramos.
# Si no existe, se mantiene como ejemplo para que el analista valide la variable correcta con el diccionario.

variable_libros = 'P5381'

if 'df' in globals() and variable_libros in df.columns:
    s = pd.to_numeric(df[variable_libros], errors='coerce')
    print(f'Variable principal: {variable_libros}')
    print('Conteo total:', len(s))
    print('Nulos:', s.isna().sum())
    print('Media:', s.mean())
    print('Mediana:', s.median())
    print('Percentiles:')
    print(s.quantile([0, 0.05, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1]).to_string())
    print('Proporción con 0 libros:', (s == 0).mean())
    print('Valores únicos principales:')
    print(s.value_counts().head(15).to_string())
else:
    print('La variable principal de libros no está disponible en esta base o aún no se validó con el diccionario.')

Variable principal: P5381
Conteo total: 96442
Nulos: 0
Media: inf
Mediana: 10.0
Percentiles:
0.00     1.0
0.05     1.0
0.25     3.0
0.50    10.0
0.75     NaN
0.90     NaN
0.95     NaN
0.99     NaN
1.00     NaN
Proporción con 0 libros: 0.0
Valores únicos principales:
P5381
inf     43172
1.0     13469
2.0      9797
3.0      8371
4.0      5555
5.0      4269
6.0      2598
10.0     2067
8.0      1393
7.0      1255
20.0      771
12.0      765
15.0      712
9.0       360
30.0      307


c:\Users\USUARIO\Desktop\Colombia_en_datos\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:4596: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a


In [15]:
# Exploración de categorías abiertas o extremas para la variable de libros.
if 'df' in globals() and 'P5381' in df.columns:
    s = pd.to_numeric(df['P5381'], errors='coerce')
    vals = s.dropna().sort_values()
    print('Valores más altos observados:')
    print(vals.tail(20).to_string())
    print('\nValores bajos observados:')
    print(vals.head(20).to_string())
    print('\nProporción de valores atípicos en extremo superior (top 1%):')
    q99 = s.quantile(0.99)
    print((s > q99).mean())
else:
    print('No hay variable de libros identificada para revisar valores atípicos.')

Valores más altos observados:
43008    inf
43010    inf
43012    inf
43013    inf
42997    inf
43014    inf
43017    inf
43018    inf
43019    inf
43023    inf
43024    inf
43035    inf
43037    inf
43039    inf
43040    inf
43041    inf
43046    inf
43047    inf
43016    inf
48220    inf

Valores bajos observados:
14065    1.0
10760    1.0
21259    1.0
51351    1.0
37339    1.0
51349    1.0
51348    1.0
72700    1.0
37342    1.0
86361    1.0
10769    1.0
78797    1.0
10772    1.0
51338    1.0
86364    1.0
51331    1.0
21248    1.0
91165    1.0
21247    1.0
51323    1.0

Proporción de valores atípicos en extremo superior (top 1%):
0.0


c:\Users\USUARIO\Desktop\Colombia_en_datos\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:4596: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a


In [16]:
# Si hay pesos, se puede comparar distribución con y sin ponderación.
# En esta exploración inicial, la idea es dejar la lógica lista para activarla cuando se confirme la variable de peso.

if 'df' in globals():
    peso_candidates = [c for c in df.columns if 'peso' in c.lower() or 'ponder' in c.lower() or 'factor' in c.lower() or 'expan' in c.lower()]
    print('Candidatas a peso en la base:', peso_candidates)
    if peso_candidates:
        for c in peso_candidates[:5]:
            print('---', c)
            print(df[c].describe())
    else:
        print('No se detectan columnas con nombres de peso explícitos en la base.')

Candidatas a peso en la base: []
No se detectan columnas con nombres de peso explícitos en la base.


## 5) Dimensión B. Frecuencia y gusto por leer

Aquí se explore la frecuencia con la que leen y el gusto por leer, para luego comprobar si la intensidad se acompaña de interés persistente o de un patrón más esporádico.

In [17]:
if 'meta' in globals():
    gusto_freq = meta[
        meta['labl'].astype(str).str.contains('gusto.*leer|le gusta.*leer|con qué frecuencia|frecuencia', case=False, na=False)
        | meta['name'].astype(str).str.contains('P1702|P1704|P1705', regex=True, na=False)
    ][['name', 'labl']].drop_duplicates()
    print('Variables de gusto y frecuencia:')
    display(gusto_freq.head(20))
else:
    print('No hay metadatos cargados.')

No hay metadatos cargados.


In [18]:
for var in ['P1702', 'P1704', 'P1705']:
    if 'df' in globals() and var in df.columns:
        s = df[var]
        print(f'\nVariable {var}:')
        print(s.value_counts(dropna=False).head(20).to_string())
        print('Nulos:', s.isna().sum())
    else:
        print(f'La variable {var} no aparece en la base.')


Variable P1702:
P1702
inf    84401
1.0     9504
2.0     2537
Nulos: 0

Variable P1704:
P1704
inf    84401
1.0     8421
2.0     3620
Nulos: 0

Variable P1705:
P1705
inf    88021
2.0     4173
1.0     3163
3.0      948
4.0      108
5.0       16
6.0       13
Nulos: 0


## 6) Dimensión C. Qué se lee

Se buscan todas las variables que describen el tipo o género de material leído, con especial cuidado para no sumar respuestas múltiples como si fueran categorías excluyentes.

In [19]:
if 'meta' in globals():
    tipo_vars = meta[
        meta['labl'].astype(str).str.contains('tipo|género|material|libro|revista|periódico|noticia|internet|digital|artículo|documento', case=False, na=False)
        | meta['name'].astype(str).str.contains('P185|P186|P187|P188|P179|P538', regex=True, na=False)
    ][['name', 'labl']].drop_duplicates()
    print('Variables potencialmente de tipo de lectura:')
    display(tipo_vars.head(100))
else:
    print('No hay metadatos cargados.')

No hay metadatos cargados.


In [20]:
# Ejemplo de análisis con respuestas múltiples (No sumar como si fueran mutuamente excluyentes)
# Se revisa la estructura de variables tipo P1852, P1853, P1854, etc.
for var in ['P1852', 'P1853', 'P1854', 'P1855', 'P1856', 'P1858', 'P1859', 'P1860', 'P1861', 'P1862']:
    if 'df' in globals() and var in df.columns:
        s = df[var]
        print(f'\n{var}:')
        print(s.value_counts(dropna=False).head(10).to_string())
        print('Nulos:', s.isna().sum())
    else:
        print(f'Variable {var} no identificada.')


P1852:
P1852
2    68872
1    27570
Nulos: 0

P1853:
P1853
2    80448
1    15994
Nulos: 0

P1854:
P1854
2    55100
1    41342
Nulos: 0

P1855:
P1855
1    63496
2    32946
Nulos: 0

P1856:
P1856
2    66613
1    29829
Nulos: 0

P1858:
P1858
2    80817
1    15625
Nulos: 0

P1859:
P1859
2    78475
1    17967
Nulos: 0

P1860:
P1860
2    50125
1    46317
Nulos: 0

P1861:
P1861
2    75417
1    21025
Nulos: 0

P1862:
P1862
2    67707
1    28735
Nulos: 0


## 7) Dimensión D. Lectura digital

Aquí se revisan materiales digitales: libros digitales, páginas web, redes sociales, blogs/foros, noticias digitales, documentos académicos y otros contenidos online.

Lo importante es comparar patrones tradicionales y digitales sin asumir sustitución ni peor calificación de unos frente a otros.

In [21]:
if 'meta' in globals():
    digital_terms = ['digital', 'web', 'redes', 'blog', 'foro', 'noticia', 'documento académico', 'internet', 'página', 'red social']
    digital_vars = meta[
        meta['labl'].astype(str).str.contains('|'.join(digital_terms), case=False, na=False)
        | meta['name'].astype(str).str.contains('P185|P186|P187|P188|P179', regex=True, na=False)
    ][['name', 'labl']].drop_duplicates()
    print('Variables relacionadas con lectura digital:')
    display(digital_vars.head(80))
else:
    print('No hay metadatos cargados.')

No hay metadatos cargados.


In [22]:
# Revisión de variables digital / frecuencia / horas dedicadas.
for var in ['P1852', 'P1853', 'P1854', 'P1855', 'P1856', 'P1858', 'P1859', 'P1860', 'P1861', 'P1862']:
    if 'df' in globals() and var in df.columns:
        s = pd.to_numeric(df[var], errors='coerce')
        print(f'\nVariable {var}:')
        print('media:', s.mean())
        print('mediana:', s.median())
        print('faltantes:', s.isna().sum())
        print('conteo de valores:', s.value_counts(dropna=False).head(10).to_string())


Variable P1852:
media: 1.7141286991144937
mediana: 2.0
faltantes: 0
conteo de valores: P1852
2    68872
1    27570

Variable P1853:
media: 1.8341593911366416
mediana: 2.0
faltantes: 0
conteo de valores: P1853
2    80448
1    15994

Variable P1854:
media: 1.5713278447149581
mediana: 2.0
faltantes: 0
conteo de valores: P1854
2    55100
1    41342

Variable P1855:
media: 1.3416146492192198
mediana: 1.0
faltantes: 0
conteo de valores: P1855
1    63496
2    32946

Variable P1856:
media: 1.6907052943738206
mediana: 2.0
faltantes: 0
conteo de valores: P1856
2    66613
1    29829

Variable P1858:
media: 1.8379855249787438
mediana: 2.0
faltantes: 0
conteo de valores: P1858
2    80817
1    15625

Variable P1859:
media: 1.8137014993467577
mediana: 2.0
faltantes: 0
conteo de valores: P1859
2    78475
1    17967

Variable P1860:
media: 1.5197424358681901
mediana: 2.0
faltantes: 0
conteo de valores: P1860
2    50125
1    46317

Variable P1861:
media: 1.7819933224113975
mediana: 2.0
faltantes: 0
con

## 8) Dimensión E. Razones para no leer

Se identifica la parte del formulario que aborda por qué no se lee o no se lee con mayor frecuencia, y se revisa si es una pregunta única o múltiple.

In [23]:
if 'meta' in globals():
    no_lee = meta[
        meta['labl'].astype(str).str.contains('razones.*no.*leer|no leyó|no lee|no leer|por qué.*no', case=False, na=False)
        | meta['name'].astype(str).str.contains('P1864|P1865|P1866|P1867|P1868|P1869|P1870|P1872|P1873', regex=True, na=False)
    ][['name', 'labl']].drop_duplicates()
    print('Variables de razones para no leer:')
    display(no_lee.head(80))
else:
    print('No hay metadatos cargados.')

No hay metadatos cargados.


In [24]:
# Revisión rápida de las variables de razones para no leer.
for var in ['P1864S1', 'P1864S2', 'P1864S3', 'P1864S4', 'P1864S5', 'P1864S6', 'P1864S7', 'P1864S8', 'P1864S9', 'P1864S10', 'P1864S11', 'P1864S12']:
    if 'df' in globals() and var in df.columns:
        s = pd.to_numeric(df[var], errors='coerce')
        print(f'\n{var}:')
        print(s.value_counts(dropna=False).head(10).to_string())
        print('faltantes:', s.isna().sum())


P1864S1:
P1864S1
inf    69166
2.0    19031
1.0     8245
faltantes: 0

P1864S2:
P1864S2
inf    69166
2.0    24429
1.0     2847
faltantes: 0

P1864S3:
P1864S3
inf    69166
2.0    23416
1.0     3860
faltantes: 0

P1864S4:
P1864S4
inf    69166
2.0    23089
1.0     4187
faltantes: 0

P1864S5:
P1864S5
inf    69166
2.0    16569
1.0    10707
faltantes: 0

P1864S6:
P1864S6
inf    69166
2.0    25603
1.0     1673
faltantes: 0

P1864S7:
P1864S7
inf    69166
2.0    20143
1.0     7133
faltantes: 0

P1864S8:
P1864S8
inf    69166
2.0    15305
1.0    11971
faltantes: 0

P1864S9:
P1864S9
inf    69166
2.0    24019
1.0     3257
faltantes: 0

P1864S10:
P1864S10
inf    69166
2.0    22578
1.0     4698
faltantes: 0

P1864S11:
P1864S11
inf    69166
2.0    20972
1.0     6304
faltantes: 0

P1864S12:
P1864S12
inf    69166
2.0    26686
1.0      590
faltantes: 0


## 9) Heterogeneidad: edad, educación, territorio y frecuencia

Sin hacer demasiados cruces, se explora si el patrón cambia por edad, educación y territorio, siempre cuidando la muestra disponible.

In [25]:
# Se busca la variable de edad, educación y zona para los cruces principales.
if 'meta' in globals():
    targets = ['edad', 'sexo', 'genero', 'educ', 'escolar', 'nivel', 'depart', 'municip', 'zona', 'territorio']
    candidates = meta[
        meta['labl'].astype(str).str.contains('|'.join(targets), case=False, na=False)
        | meta['name'].astype(str).str.contains('P17|P20|P21|P22|P23|P24|P25|P26|P27|P28', regex=True, na=False)
    ][['name', 'labl']].drop_duplicates()
    print('Candidatas para edad, educación y territorio:')
    display(candidates.head(80))
else:
    print('No hay metadatos cargados.')

No hay metadatos cargados.


In [26]:
# Próximo análisis recomendado en la práctica:
# 1. definir la variable de edad real (del diccionario); 
# 2. definir la variable de educación real (del diccionario);
# 3. definir la variable de territorio real (si existe en la base o en otra tabla complementaria);
# 4. calcular medias / proporciones por grupos usando la ponderación correcta cuando exista.

print('Checklist de cruces a ejecutar una vez se validen las variables exactas:')
print('- edad x libros leídos')
print('- nivel educativo x libros leídos')
print('- edad x frecuencia de lectura')
print('- nivel educativo x frecuencia de lectura')
print('- territorio x intensidad de lectura (si la muestra permite la desagregación)')

Checklist de cruces a ejecutar una vez se validen las variables exactas:
- edad x libros leídos
- nivel educativo x libros leídos
- edad x frecuencia de lectura
- nivel educativo x frecuencia de lectura
- territorio x intensidad de lectura (si la muestra permite la desagregación)


## 10) Qué debe entregarse al final de la exploración

Después de este análisis, el producto final no debería ser una conclusión causal ni una publicación, sino una síntesis descriptiva clara.

### Entregables esperados
1. Las 10 variables más importantes para estudiar lectura.
2. Los 5 hallazgos descriptivos más interesantes.
3. Los 3 patrones que merecen una investigación posterior.
4. Las 3 cosas que NO podemos concluir con esta base.
5. Cualquier problema metodológico importante antes de usar estos datos en un video.

### Reglas de cuidado
- No confundir número de libros con capacidad lectora.
- No afirmar que el celular causa menor lectura.
- No concluir que lectura digital es peor que lectura física.
- Usar factores de expansión cuando correspondan.
- Mantener visibles los valores faltantes y no aplica.

In [ ]:
# Plantilla breve para resumir los hallazgos al final del trabajo.
# Se deja lista para completar con la evidencia real del análisis.
summary = {
    'variables_clave': [],
    'hallazgos_descriptivos': [],
    'patrones_para_investigar': [],
    'no_se_puede_concluir': [],
    'problemas_metodologicos': []
}
print(summary)

# Exploración inicial con diccionario de datos. codigo-significado.

In [31]:
from pathlib import Path

print(Path.cwd())

c:\Users\USUARIO\Desktop\Colombia_en_datos\episodios\005_pruebas_pisa\notebooks


In [32]:
from pathlib import Path

print(list(Path.cwd().iterdir()))

[WindowsPath('c:/Users/USUARIO/Desktop/Colombia_en_datos/episodios/005_pruebas_pisa/notebooks/001_exploracion_inicial.ipynb'), WindowsPath('c:/Users/USUARIO/Desktop/Colombia_en_datos/episodios/005_pruebas_pisa/notebooks/README.md')]


In [33]:
from pathlib import Path

proyecto = Path.home() / "Desktop" / "Colombia_en_datos"

archivos = list(proyecto.rglob("ENLEC_FORM_LECTURA.csv"))

for archivo in archivos:
    print(archivo)

C:\Users\USUARIO\Desktop\Colombia_en_datos\episodios\005_pruebas_pisa\data\raw\ENLEC_FORM_LECTURA.csv


In [34]:
archivos_xml = list(proyecto.rglob("DANE-DIMPE-ENLEC-2017.xml"))

for archivo in archivos_xml:
    print(archivo)

C:\Users\USUARIO\Desktop\Colombia_en_datos\episodios\005_pruebas_pisa\data\raw\DANE-DIMPE-ENLEC-2017.xml


In [35]:
from pathlib import Path
import pandas as pd
import xml.etree.ElementTree as ET

# Carpeta de datos originales
RAW_DIR = Path(
    r"C:\Users\USUARIO\Desktop\Colombia_en_datos\episodios\005_pruebas_pisa\data\raw"
)

# Archivos
archivo_datos = RAW_DIR / "ENLEC_FORM_LECTURA.csv"
archivo_diccionario = RAW_DIR / "DANE-DIMPE-ENLEC-2017.xml"

# Verificación
print("CSV:", archivo_datos.exists())
print("XML:", archivo_diccionario.exists())

CSV: True
XML: True


In [36]:
# Cargar base
df = pd.read_csv(
    archivo_datos,
    low_memory=False
)

print(f"Base cargada: {df.shape[0]:,} filas x {df.shape[1]} columnas")

Base cargada: 96,442 filas x 163 columnas


In [37]:
# Cargar diccionario XML
tree = ET.parse(archivo_diccionario)
root = tree.getroot()

print("Diccionario cargado correctamente.")

Diccionario cargado correctamente.


# inspeccionemos el XML

In [38]:
print(root.tag)
print(len(root))

{http://www.icpsr.umich.edu/DDI}codeBook
9


In [39]:
for elemento in root:
    print(elemento.tag, elemento.attrib)

{http://www.icpsr.umich.edu/DDI}docDscr {}
{http://www.icpsr.umich.edu/DDI}stdyDscr {}
{http://www.icpsr.umich.edu/DDI}fileDscr {'ID': 'F1', 'URI': 'ENLEC_2017_25-07-2018.Ajust.Nesstar?Index=0&Name=ENLEC_ESCRITURA_BIBLIOTECAS'}
{http://www.icpsr.umich.edu/DDI}fileDscr {'ID': 'F2', 'URI': 'ENLEC_2017_25-07-2018.Ajust.Nesstar?Index=1&Name=ENLEC_FORM_LECTURA'}
{http://www.icpsr.umich.edu/DDI}fileDscr {'ID': 'F3', 'URI': 'ENLEC_2017_25-07-2018.Ajust.Nesstar?Index=2&Name=ENLEC_CERO_A_CUATRO'}
{http://www.icpsr.umich.edu/DDI}fileDscr {'ID': 'F4', 'URI': 'ENLEC_2017_25-07-2018.Ajust.Nesstar?Index=3&Name=ENLEC_ADMIN_PERSONAS'}
{http://www.icpsr.umich.edu/DDI}fileDscr {'ID': 'F5', 'URI': 'ENLEC_2017_25-07-2018.Ajust.Nesstar?Index=4&Name=ENLEC_ADMIN_VIVIENDA'}
{http://www.icpsr.umich.edu/DDI}fileDscr {'ID': 'F6', 'URI': 'ENLEC_2017_25-07-2018.Ajust.Nesstar?Index=5&Name=ENLEC_ADMIN_HOGAR'}
{http://www.icpsr.umich.edu/DDI}dataDscr {}


# encontrar las variables

In [40]:
# Namespace del XML
ns = {"ddi": "http://www.icpsr.umich.edu/DDI"}

# Buscar todas las variables descritas en el diccionario
variables_xml = root.findall(".//ddi:dataDscr/ddi:var", ns)

print(f"Variables encontradas en el XML: {len(variables_xml)}")

# Mostrar las primeras 10
for var in variables_xml[:10]:
    print(var.attrib)

Variables encontradas en el XML: 355
{'ID': 'V1', 'name': 'FK_ID_FORMULARIO', 'files': 'F1', 'dcml': '0', 'intrvl': 'discrete'}
{'ID': 'V2', 'name': 'ID_VIVIENDA', 'files': 'F1', 'dcml': '0', 'intrvl': 'discrete'}
{'ID': 'V3', 'name': 'NUMERO_HOGAR', 'files': 'F1', 'dcml': '0', 'intrvl': 'discrete'}
{'ID': 'V4', 'name': 'NUMERO_PERSONA', 'files': 'F1', 'dcml': '0', 'intrvl': 'discrete'}
{'ID': 'V5', 'name': 'P1843S1', 'files': 'F1', 'dcml': '0', 'intrvl': 'discrete'}
{'ID': 'V6', 'name': 'P1843S2', 'files': 'F1', 'dcml': '0', 'intrvl': 'discrete'}
{'ID': 'V7', 'name': 'P1843S3', 'files': 'F1', 'dcml': '0', 'intrvl': 'discrete'}
{'ID': 'V8', 'name': 'P1843S4', 'files': 'F1', 'dcml': '0', 'intrvl': 'discrete'}
{'ID': 'V9', 'name': 'P1843S6', 'files': 'F1', 'dcml': '0', 'intrvl': 'discrete'}
{'ID': 'V10', 'name': 'P1843S7', 'files': 'F1', 'dcml': '0', 'intrvl': 'discrete'}


# buscar directamente nuestras tres variables

In [41]:
variables_objetivo = ["P1704", "P1705", "P5381"]

for codigo in variables_objetivo:
    encontrados = [
        var for var in variables_xml
        if var.attrib.get("name") == codigo
        or var.attrib.get("ID") == codigo
    ]

    print(f"\n{codigo}: {len(encontrados)} encontrada(s)")

    for var in encontrados:
        print(var.attrib)


P1704: 1 encontrada(s)
{'ID': 'V91', 'name': 'P1704', 'files': 'F2', 'dcml': '0', 'intrvl': 'discrete'}

P1705: 1 encontrada(s)
{'ID': 'V92', 'name': 'P1705', 'files': 'F2', 'dcml': '0', 'intrvl': 'discrete'}

P5381: 1 encontrada(s)
{'ID': 'V132', 'name': 'P5381', 'files': 'F2', 'dcml': '0', 'intrvl': 'contin'}


In [42]:
def inspeccionar_variable(codigo):
    """
    Muestra la información disponible en el diccionario DDI
    para una variable específica.
    """
    
    variables = [
        var for var in variables_xml
        if var.attrib.get("name") == codigo
    ]
    
    if not variables:
        print(f"No se encontró la variable {codigo}")
        return
    
    var = variables[0]
    
    print("=" * 70)
    print(f"VARIABLE: {codigo}")
    print("=" * 70)
    
    print("\nATRIBUTOS:")
    for clave, valor in var.attrib.items():
        print(f"  {clave}: {valor}")
    
    print("\nCONTENIDO:")
    
    for elemento in var:
        print(f"\n[{elemento.tag.split('}')[-1]}]")
        print("Atributos:", elemento.attrib)
        
        texto = "".join(elemento.itertext()).strip()
        
        if texto:
            print("Texto:", texto[:1000])

In [43]:
inspeccionar_variable("P1704")

VARIABLE: P1704

ATRIBUTOS:
  ID: V91
  name: P1704
  files: F2
  dcml: 0
  intrvl: discrete

CONTENIDO:

[location]
Atributos: {'StartPos': '12', 'EndPos': '12', 'width': '1', 'RecSegNo': '1'}

[labl]
Atributos: {}
Texto: ¿A .. le gusta leer?

[security]
Atributos: {}
Texto: El acceso a microdatos y Mam-up se considera como de tratamiento especial respecto a la reserva estadística por tanto estará sujeto a la reglamentación que para el efecto defina el Comité de Aseguramiento de la reserva estadística. Resolución 173 de 2008

[respUnit]
Atributos: {}
Texto: Informante directo: Personas de 12 años y más.

Informante idóneo: Personas de 0 a 11 años.

[qstn]
Atributos: {}
Texto: 2. A ¿quién prefiere que le lea?
        
        
          3. ¿A .. le gusta leer?

1. Sí
2. No
        
        
          3.a. Con qué frecuencia:

[valrng]
Atributos: {}

[universe]
Atributos: {'clusion': 'I'}
Texto: Está conformado por la población civil no institucional residente en todo el territorio naci

In [44]:
inspeccionar_variable("P1705")

VARIABLE: P1705

ATRIBUTOS:
  ID: V92
  name: P1705
  files: F2
  dcml: 0
  intrvl: discrete

CONTENIDO:

[location]
Atributos: {'StartPos': '13', 'EndPos': '13', 'width': '1', 'RecSegNo': '1'}

[labl]
Atributos: {}
Texto: Con qué frecuencia:

[security]
Atributos: {}
Texto: El acceso a microdatos y Mam-up se considera como de tratamiento especial respecto a la reserva estadística por tanto estará sujeto a la reglamentación que para el efecto defina el Comité de Aseguramiento de la reserva estadística. Resolución 173 de 2008

[respUnit]
Atributos: {}
Texto: Informante directo: Personas de 12 años y más.

Informante idóneo: Personas de 0 a 11 años.

[qstn]
Atributos: {}
Texto: 3. ¿A .. le gusta leer?
        
        
          3.a. Con qué frecuencia:

1. Todos los días
2. Varias veces a la semana
3. Una vez a la semana
4. Una vez al mes 
5. Una vez cada tres meses
6. Por lo menos una vez al año
        
        
          4. En los últimos 12 meses ¿leyó artículos o documentos académi

In [45]:
inspeccionar_variable("P5381")

VARIABLE: P5381

ATRIBUTOS:
  ID: V132
  name: P5381
  files: F2
  dcml: 0
  intrvl: contin

CONTENIDO:

[location]
Atributos: {'StartPos': '66', 'EndPos': '68', 'width': '3', 'RecSegNo': '1'}

[labl]
Atributos: {}
Texto: En los últimos 12 meses, ¿cuántos libros leyó o le leyeron?

[security]
Atributos: {}
Texto: El acceso a microdatos y Mam-up se considera como de tratamiento especial respecto a la reserva estadística por tanto estará sujeto a la reglamentación que para el efecto defina el Comité de Aseguramiento de la reserva estadística. Resolución 173 de 2008

[respUnit]
Atributos: {}
Texto: Informante directo: Personas de 12 años y más.

Informante idóneo: Personas de 0 a 11 años.

[qstn]
Atributos: {}
Texto: 16. En los últimos 12 meses ¿... leyó documentos académicos impresos?
        
        
          17. En los últimos 12 meses, ¿cuántos libros leyó o le leyeron?
        
        
          17.a. De esos libros, ¿cuántos eran textos escolares o de estudio?

[valrng]
Atributos

# primera exploración

In [46]:
# ============================================================
# PRIMERA EXPLORACIÓN
# ============================================================

variables = ["P1704", "P1705", "P5381"]

for variable in variables:
    print("\n" + "=" * 70)
    print(variable)
    print("=" * 70)
    
    print("\nTipo:")
    print(df[variable].dtype)
    
    print("\nValores únicos:")
    print(df[variable].nunique(dropna=False))
    
    print("\nFrecuencias:")
    print(df[variable].value_counts(dropna=False).sort_index())


P1704

Tipo:
float64

Valores únicos:
3

Frecuencias:
P1704
1.0     8421
2.0     3620
inf    84401
Name: count, dtype: int64

P1705

Tipo:
float64

Valores únicos:
7

Frecuencias:
P1705
1.0     3163
2.0     4173
3.0      948
4.0      108
5.0       16
6.0       13
inf    88021
Name: count, dtype: int64

P5381

Tipo:
float64

Valores únicos:
88

Frecuencias:
P5381
1.0      13469
2.0       9797
3.0       8371
4.0       5555
5.0       4269
         ...  
180.0        1
200.0        9
207.0        1
250.0       19
inf      43172
Name: count, Length: 88, dtype: int64


In [47]:
pd.crosstab(
    df["P1704"],
    df["P1705"],
    dropna=False
)

P1705,1.0,2.0,3.0,4.0,5.0,6.0,inf
P1704,,,,,,,
1.0,3163,4173,948,108,16,13,0
2.0,0,0,0,0,0,0,3620
inf,0,0,0,0,0,0,84401


In [48]:
print(
    df.groupby(["P1704", "P1705"], dropna=False)
      .size()
      .sort_index()
)

P1704  P1705
1.0    1.0       3163
       2.0       4173
       3.0        948
       4.0        108
       5.0         16
       6.0         13
2.0    inf       3620
inf    inf      84401
dtype: int64


In [49]:
pd.crosstab(
    df["P1704"],
    df["P5381"],
    dropna=False
)

P5381,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0,9.0,10.0,...,140.0,144.0,150.0,166.0,170.0,180.0,200.0,207.0,250.0,inf
P1704,,,,,,,,,,,,,,,,,,,,,
1.0,1026,1444,1189,843,635,354,182,179,46,249,...,0,0,1,0,0,1,0,0,3,1921
2.0,340,326,259,153,105,43,27,30,8,30,...,0,0,1,0,0,0,1,0,0,2258
inf,12103,8027,6923,4559,3529,2201,1046,1184,306,1788,...,1,2,5,1,1,0,8,1,16,38993


In [50]:
df.groupby("P1704")["P5381"].agg(
    total="size",
    validos=lambda x: x.notna().sum(),
    inf=lambda x: (x == float("inf")).sum(),
    minimo=lambda x: x[x != float("inf")].min(),
    maximo=lambda x: x[x != float("inf")].max()
)

,total,validos,inf,minimo,maximo
P1704,,,,,
1.0,8421,8421,1921,1.0,250.0
2.0,3620,3620,2258,1.0,200.0
inf,84401,84401,38993,1.0,250.0


In [51]:
tabla = (
    df.groupby("P1704")["P5381"]
      .agg(
          casos="size",
          con_dato=lambda x: (~x.isin([float("inf")])).sum(),
          sin_dato=lambda x: (x == float("inf")).sum()
      )
)

tabla

,casos,con_dato,sin_dato
P1704,,,
1.0,8421,6500,1921
2.0,3620,1362,2258
inf,84401,45408,38993


In [52]:
serie = df["P5381"]

print("Valores < 1:")
print(serie[(serie < 1) & (serie != float("inf"))].value_counts())

print("\nValores > 250:")
print(serie[serie > 250].value_counts())

print("\nValores entre 1 y 250:")
print(serie[(serie >= 1) & (serie <= 250)].count())

Valores < 1:
Series([], Name: count, dtype: int64)

Valores > 250:
P5381
inf    43172
Name: count, dtype: int64

Valores entre 1 y 250:
53270


In [53]:
# Buscar referencias al valor especial y a valores inválidos/missing
texto_xml = ET.tostring(root, encoding="unicode")

for termino in [
    "1.79769313486232",
    "missing",
    "invald",
    "invalid",
    "inval",
    "No informa",
    "No sabe"
]:
    print(f"\n--- {termino} ---")
    posiciones = [i for i in range(len(texto_xml)) if texto_xml.startswith(termino, i)]
    print("Coincidencias:", len(posiciones))


--- 1.79769313486232 ---
Coincidencias: 0

--- missing ---
Coincidencias: 0

--- invald ---
Coincidencias: 0

--- invalid ---
Coincidencias: 0

--- inval ---
Coincidencias: 0

--- No informa ---
Coincidencias: 29

--- No sabe ---
Coincidencias: 36


In [54]:
inspeccionar_variable("P5381")

VARIABLE: P5381

ATRIBUTOS:
  ID: V132
  name: P5381
  files: F2
  dcml: 0
  intrvl: contin

CONTENIDO:

[location]
Atributos: {'StartPos': '66', 'EndPos': '68', 'width': '3', 'RecSegNo': '1'}

[labl]
Atributos: {}
Texto: En los últimos 12 meses, ¿cuántos libros leyó o le leyeron?

[security]
Atributos: {}
Texto: El acceso a microdatos y Mam-up se considera como de tratamiento especial respecto a la reserva estadística por tanto estará sujeto a la reglamentación que para el efecto defina el Comité de Aseguramiento de la reserva estadística. Resolución 173 de 2008

[respUnit]
Atributos: {}
Texto: Informante directo: Personas de 12 años y más.

Informante idóneo: Personas de 0 a 11 años.

[qstn]
Atributos: {}
Texto: 16. En los últimos 12 meses ¿... leyó documentos académicos impresos?
        
        
          17. En los últimos 12 meses, ¿cuántos libros leyó o le leyeron?
        
        
          17.a. De esos libros, ¿cuántos eran textos escolares o de estudio?

[valrng]
Atributos

In [55]:
var = [
    v for v in variables_xml
    if v.attrib.get("name") == "P5381"
][0]

for elemento in var.iter():
    print(
        elemento.tag.split("}")[-1],
        elemento.attrib,
        repr(" ".join(elemento.itertext()).strip())
    )

var {'ID': 'V132', 'name': 'P5381', 'files': 'F2', 'dcml': '0', 'intrvl': 'contin'} 'En los últimos 12 meses, ¿cuántos libros leyó o le leyeron?\n       \n       \n        El acceso a microdatos y Mam-up se considera como de tratamiento especial respecto a la reserva estadística por tanto estará sujeto a la reglamentación que para el efecto defina el Comité de Aseguramiento de la reserva estadística. Resolución 173 de 2008\n       \n       \n        Informante directo: Personas de 12 años y más.\n\nInformante idóneo: Personas de 0 a 11 años.\n       \n       \n         \n          16. En los últimos 12 meses ¿... leyó documentos académicos impresos?\n         \n         \n          17. En los últimos 12 meses, ¿cuántos libros leyó o le leyeron?\n         \n         \n          17.a. De esos libros, ¿cuántos eran textos escolares o de estudio?\n         \n       \n       \n         \n       \n       \n        Está conformado por la población civil no institucional residente en todo el t

In [56]:
for variable in ["P1704", "P1705", "P5381"]:
    print("\n", "=" * 50)
    print(variable)
    
    print("inf:", np.isinf(df[variable]).sum())
    print("NaN:", df[variable].isna().sum())
    
    valores_inf = df.loc[np.isinf(df[variable]), variable]
    print("Ejemplo de valores inf:")
    print(valores_inf.head())


P1704
inf: 84401
NaN: 0
Ejemplo de valores inf:
0    inf
1    inf
3    inf
5    inf
6    inf
Name: P1704, dtype: float64

P1705
inf: 88021
NaN: 0
Ejemplo de valores inf:
0    inf
1    inf
2    inf
3    inf
4    inf
Name: P1705, dtype: float64

P5381
inf: 43172
NaN: 0
Ejemplo de valores inf:
0    inf
1    inf
2    inf
4    inf
5    inf
Name: P5381, dtype: float64


In [57]:
print(df.loc[np.isinf(df["P5381"]), ["P1704", "P1705", "P5381"]].head(20))

    P1704  P1705  P5381
0     inf    inf    inf
1     inf    inf    inf
2     2.0    inf    inf
4     2.0    inf    inf
5     inf    inf    inf
7     inf    inf    inf
12    inf    inf    inf
13    inf    inf    inf
14    inf    inf    inf
15    inf    inf    inf
17    inf    inf    inf
18    inf    inf    inf
21    inf    inf    inf
23    inf    inf    inf
24    inf    inf    inf
25    1.0    2.0    inf
28    inf    inf    inf
29    inf    inf    inf
35    inf    inf    inf
45    inf    inf    inf


In [58]:
df_limpio = df.copy()

for variable in ["P1704", "P1705", "P5381"]:
    df_limpio.loc[np.isinf(df_limpio[variable]), variable] = np.nan

# limpieza 
sin tocar los datos crudos.

In [59]:
df_limpio = df.copy()

# Convertir inf en NaN únicamente en las variables que estamos estudiando
variables_lectura = ["P1704", "P1705", "P5381"]

for variable in variables_lectura:
    df_limpio[variable] = df_limpio[variable].replace(
        [np.inf, -np.inf],
        np.nan
    )

# Comprobación
print(df_limpio[variables_lectura].isna().sum())

P1704    84401
P1705    88021
P5381    43172
dtype: int64


In [60]:
for variable in variables_lectura:
    print("\n", variable)
    print(df_limpio[variable].value_counts(dropna=False).sort_index())


 P1704
P1704
1.0     8421
2.0     3620
NaN    84401
Name: count, dtype: int64

 P1705
P1705
1.0     3163
2.0     4173
3.0      948
4.0      108
5.0       16
6.0       13
NaN    88021
Name: count, dtype: int64

 P5381
P5381
1.0      13469
2.0       9797
3.0       8371
4.0       5555
5.0       4269
         ...  
180.0        1
200.0        9
207.0        1
250.0       19
NaN      43172
Name: count, Length: 88, dtype: int64


Para una encuesta como ENLEC, necesitamos saber si existe un peso/factor de expansión y cuál corresponde a F2.

Vamos a buscarlo sistemáticamente en los nombres de las 163 columnas que realmente tenemos.

In [61]:
palabras_clave = [
    "factor",
    "expan",
    "peso",
    "ponder",
    "elev",
    "fac",
    "fex",
    "weight"
]

columnas_factor = [
    col for col in df.columns
    if any(palabra in col.lower() for palabra in palabras_clave)
]

print("Columnas potencialmente relacionadas con ponderación:")
for col in columnas_factor:
    print(col)

Columnas potencialmente relacionadas con ponderación:


In [62]:
terminos = [
    "factor",
    "expan",
    "peso",
    "ponder",
    "elev",
    "weight"
]

for termino in terminos:
    encontrados = [
        var.attrib.get("name")
        for var in variables_xml
        if termino.lower() in ET.tostring(
            var, encoding="unicode"
        ).lower()
    ]
    
    print(f"\n{termino}: {encontrados}")


factor: ['DOMINIO', 'FEX_C']

expan: ['DOMINIO', 'FEX_C']

peso: []

ponder: []

elev: ['P1066S4', 'P1066S6', 'P1066S10', 'P1787', 'P1872S4', 'P1872S5', 'P1872S10', 'P1876S7A1', 'P1876S8A1', 'P1702S1', 'P1872S1A5', 'P1872S1A6', 'P1872S1A7', 'P201S25A1', 'P201S2', 'P201S3']

weight: []


inspeccionemos FEX_C

In [63]:
inspeccionar_variable("FEX_C")

VARIABLE: FEX_C

ATRIBUTOS:
  ID: V326
  name: FEX_C
  files: F5
  dcml: 2
  intrvl: contin

CONTENIDO:

[location]
Atributos: {'StartPos': '41', 'EndPos': '47', 'width': '7', 'RecSegNo': '1'}

[labl]
Atributos: {}
Texto: Factor de Expansión

[security]
Atributos: {}
Texto: El acceso a microdatos y Mam-up se considera como de tratamiento especial respecto a la reserva estadística por tanto estará sujeto a la reglamentación que para el efecto defina el Comité de Aseguramiento de la reserva estadística. Resolución 173 de 2008

[qstn]
Atributos: {}
Texto: Union de Clases
        
        
          Factor de Expansión

[valrng]
Atributos: {}

[sumStat]
Atributos: {'type': 'vald'}
Texto: 0

[sumStat]
Atributos: {'type': 'invd'}
Texto: 0

[varFormat]
Atributos: {'type': 'numeric', 'schema': 'other'}


In [65]:
# Buscar variables relacionadas con factores de expansión
# que pertenezcan específicamente a F2

for var in variables_xml:
    nombre = var.attrib.get("name", "")
    archivo = var.attrib.get("files", "")
    
    texto = " ".join(
        " ".join(elemento.itertext())
        for elemento in var
    ).lower()
    
    if archivo == "F2" and (
        "factor de expansión" in texto
        or "factor de expansion" in texto
        or "expansión" in texto
        or "expansion" in texto
    ):
        print("=" * 60)
        print(var.attrib)
        print(texto[:500])

In [66]:
for var in variables_xml:
    if var.attrib.get("files") == "F2":
        texto = " ".join(
            " ".join(elemento.itertext())
            for elemento in var
        ).lower()
        
        if "factor" in texto or "expan" in texto:
            print(var.attrib)
            print(texto[:800])
            print()

In [67]:
columnas_id = [
    col for col in df.columns
    if any(x in col.upper() for x in [
        "ID", "VIVIENDA", "HOGAR", "PERSONA", "NUMERO"
    ])
]

for col in columnas_id:
    print(col)

FK_ID_FORMULARIO
ID_VIVIENDA
NUMERO_HOGAR
NUMERO_PERSONA


In [68]:
for archivo in RAW_DIR.iterdir():
    print(archivo.name)

.gitkeep
DANE-DIMPE-ENLEC-2017.xml
ENLEC_FORM_LECTURA.csv
README.md
variables-09-09-26-100232.csv
variables-09-09-26-100316.csv
variables-09-09-26-100327.csv
variables-09-09-26-100336.csv
variables-09-09-26-100346.csv
variables-09-09-26-100356.csv
variables-09-09-26-100454.csv
variables-09-09-26-100531.csv


In [69]:
archivos_metadata = list(RAW_DIR.glob("variables-*.csv"))

for archivo in archivos_metadata:
    temp = pd.read_csv(archivo, low_memory=False)
    
    print("\n" + "=" * 80)
    print(archivo.name)
    print("=" * 80)
    print("Dimensiones:", temp.shape)
    print("Columnas:", temp.columns.tolist())
    print(temp.head(3))


variables-09-09-26-100232.csv
Dimensiones: (1, 27)
Columnas: ['survey_idno', 'sid', 'file_id', 'vid', 'name', 'labl', 'var_intrvl', 'var_dcml', 'var_wgt', 'var_start_pos', 'var_end_pos', 'var_width', 'var_imputation', 'var_security', 'var_respunit', 'var_qstn_preqtxt', 'var_qstn_qstnlit', 'var_qstn_postqtxt', 'var_qstn_ivulnstr', 'var_universe', 'var_sumstat', 'var_txt', 'var_catgry', 'var_codinstr', 'var_concept', 'var_format', 'var_notes']
             survey_idno  sid file_id  vid   name  \
0  DANE-DIMPE-ENLEC-2017  550      F2  V89  P1702   

                           labl var_intrvl  var_dcml  var_wgt  var_start_pos  \
0  A ¿.. ¿le gusta que le lean?   discrete         0      NaN            NaN   

   ...                 var_qstn_postqtxt  var_qstn_ivulnstr  \
0  ...  2. A ¿quién prefiere que le lea?                NaN   

                                        var_universe  \
0  Está conformado por la población civil no inst...   

                                         var_

In [70]:
for archivo in archivos_metadata:
    temp = pd.read_csv(archivo, low_memory=False)
    
    texto = temp.astype(str).to_string().lower()
    
    if "fex_c" in texto or "factor de expansión" in texto or "factor de expansion" in texto:
        print("\nENCONTRADO EN:", archivo.name)
        print(
            temp[
                temp.astype(str)
                    .apply(
                        lambda col: col.str.contains(
                            "FEX_C|factor de expansión|factor de expansion",
                            case=False,
                            na=False
                        )
                    )
                    .any(axis=1)
            ]
        )

tabla de cobertura de variables

In [71]:
variables = ["P1704", "P1705", "P5381"]

for variable in variables:
    validos = df_limpio[variable].notna().sum()
    total = len(df_limpio)
    
    print(
        f"{variable}: "
        f"{validos:,} válidos de {total:,} "
        f"({validos / total * 100:.2f}%)"
    )

P1704: 12,041 válidos de 96,442 (12.49%)
P1705: 8,421 válidos de 96,442 (8.73%)
P5381: 53,270 válidos de 96,442 (55.24%)


In [72]:
df_limpio["P5381"].describe(percentiles=[
    0.25, 0.50, 0.75, 0.90, 0.95, 0.99
])

count    53270.000000
mean         4.940023
std          9.298928
min          1.000000
25%          1.000000
50%          3.000000
75%          5.000000
90%         10.000000
95%         15.000000
99%         36.000000
max        250.000000
Name: P5381, dtype: float64

In [73]:
dist_libros = (
    df_limpio["P5381"]
    .value_counts()
    .sort_index()
)

dist_libros.head(30)

P5381
1.0     13469
2.0      9797
3.0      8371
4.0      5555
5.0      4269
6.0      2598
7.0      1255
8.0      1393
9.0       360
10.0     2067
11.0      164
12.0      765
13.0      142
14.0      114
15.0      712
16.0       82
17.0       51
18.0       67
19.0       19
20.0      771
21.0       16
22.0       22
23.0       17
24.0       79
25.0      160
26.0       13
27.0       13
28.0       12
29.0        4
30.0      307
Name: count, dtype: int64

In [74]:
print("1 libro:", (df_limpio["P5381"] == 1).sum())
print("2 libros:", (df_limpio["P5381"] == 2).sum())
print("3 libros:", (df_limpio["P5381"] == 3).sum())
print("4 libros:", (df_limpio["P5381"] == 4).sum())
print("5 libros:", (df_limpio["P5381"] == 5).sum())

print("\nMás de 10:", (df_limpio["P5381"] > 10).sum())
print("Más de 20:", (df_limpio["P5381"] > 20).sum())
print("Más de 50:", (df_limpio["P5381"] > 50).sum())
print("Más de 100:", (df_limpio["P5381"] > 100).sum())

1 libro: 13469
2 libros: 9797
3 libros: 8371
4 libros: 5555
5 libros: 4269

Más de 10: 4136
Más de 20: 1249
Más de 50: 239
Más de 100: 56


In [75]:
(df_limpio["P5381"] == 250).sum()

np.int64(19)